 The important changes are:

  - INT8_KEEP_FLOAT_MAX_NUMEL increased from 65536 to 131072
      - More small/medium tensors stay in float instead of being forced into int8.
      - This usually improves final roundtrip val_loss and val_bpb, at the cost of larger artifact size.
  - INT8_CLIP_PERCENTILE default nudged higher from 99.99984 to 99.99995
      - This clips fewer outliers before int8 quantization.
      - It is a small quality-vs-compression tradeoff tweak.
  - EXPORT_COMPRESSOR=lzma
      - This does not improve model quality.
      - It only tries to shrink the final file more aggressively than zlib.
  - self-generated calibration search
      - This is the real new idea.
      - Instead of hardcoding one clip percentile, the script generates its own token sequences, quantizes the model several ways, scores those candidates on those generated sequences, and picks the
        best clip percentile before writing the final artifact.
      - That logic is in colab/2026-04-06_QuantExport2_selfgenerated_calib/train_gpt.py:880 and colab/2026-04-06_QuantExport2_selfgenerated_calib/train_gpt.py:963.
      - SELF_CALIB_VARIANTS=ar_sample,ar_greedy,ar_topk
      - SELF_CALIB_NUM_SEQS=24
      - SELF_CALIB_SEQ_LEN=512
      - SELF_CALIB_BATCH_SIZE=4
      - SELF_CALIB_TEMPERATURE=0.8
      - SELF_CALIB_TOPK=32
      - SELF_CALIB_CANDIDATE_PERCENTILES=99.999,99.9995,99.99984,99.99995,100.0

  Calibration here means: choose quantization settings using inputs that are representative of how the model behaves.

  In this experiment, those inputs are not taken from train or validation files. They are generated by the model itself.

  The process is:

  1. Train the model normally.
  2. Ask the model to generate short token sequences autoregressively.
  3. Try several quantization candidates, mainly different INT8_CLIP_PERCENTILE values.
  4. For each candidate:
      - quantize the weights
      - dequantize them back
      - run the quantized/dequantized model on the generated sequences
      - measure the average loss on those sequences
  5. Pick the clip percentile with the best calibration loss, with compressed size as tie-break.

  That is implemented in colab/2026-04-06_QuantExport2_selfgenerated_calib/train_gpt.py:963.

  So self-generated calibration means:

  - the model manufactures its own calibration set
  - the calibration set is used only to choose export settings
  - there is no extra training
  - there is no access to validation data for this search

  What the self-calibration env vars mean
  From colab/2026-04-06_QuantExport2_selfgenerated_calib/run.sh:89:

  - SELF_CALIB_VARIANTS
      - which decoding styles are used to generate calibration tokens
      - ar_greedy: always pick argmax
      - ar_sample: sample from full distribution
      - ar_topk: sample from top-k only
      - ar_mixed: softer hybrid
  - SELF_CALIB_NUM_SEQS
      - how many generated sequences to use
  - SELF_CALIB_SEQ_LEN
      - length of each sequence
  - SELF_CALIB_BATCH_SIZE
      - generation/eval batch size
  - SELF_CALIB_TEMPERATURE
      - randomness level for sampled variants
  - SELF_CALIB_TOPK
      - top-k cutoff for ar_topk
  - SELF_CALIB_CANDIDATE_PERCENTILES
      - the clip-percentile values to test during export search


In [ ]:
!git clone https://github.com/IanniMuliterno/parameter-golf.git


Cloning into 'parameter-golf'...
remote: Enumerating objects: 659, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 659 (delta 7), reused 4 (delta 4), pack-reused 650 (from 3)
Receiving objects: 100% (659/659), 1.44 MiB | 4.14 MiB/s, done.
Resolving deltas: 100% (285/285), done.


In [ ]:
#!cd /content/parameter-golf && git pull

/bin/bash: line 1: cd: /content/parameter-golf: No such file or directory


In [ ]:
%cd parameter-golf/colab/2026-04-06_QuantExport2_selfgenerated_calib

/content/parameter-golf/colab/2026-04-06_QuantExport2_selfgenerated_calib


In [ ]:
!INSTALL_DEPS=1 ENABLE_MATH_SDP=1 VAL_LOSS_EVERY=0 TRAIN_LOG_EVERY=1 ITERATIONS=300 bash run.sh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.8 MB/s eta 0:00:00
manifest.json: 1.93kB [00:00, 7.43MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 124M/124M [00:01<00:00, 67.0MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:01<00:00, 124MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:01<00:00, 124MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:02<00:00, 99.5MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:01<00:00, 124MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:01<00:00, 110MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:02<00:00, 99.5MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:01<00:00, 124MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:04<00:00, 45.4MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:02<00:00, 99.5MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 10